# Enterprise Phase 1: Data Understanding & EDA

This notebook is organized as a professional analytics workflow with clear sections, reusable code, and business interpretations.


## 1. Imports



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize']=(10,5)
plt.rcParams['axes.grid']=True

DATA_PATH='data/raw/american_bankruptcy.csv'
df=pd.read_csv(DATA_PATH)


## 2. Utility Functions



In [ ]:
def dataset_summary(df):
    return pd.DataFrame({
        'Metric':['Rows','Columns','Missing Values','Duplicate Rows','Companies','Years'],
        'Value':[len(df),df.shape[1],int(df.isna().sum().sum()),
                 int(df.duplicated().sum()),
                 df['company_name'].nunique(),
                 f"{df['year'].min()}-{df['year'].max()}"]
    })

def missing_report(df):
    r=df.isna().sum().to_frame('Missing')
    r['Percent']=100*r['Missing']/len(df)
    return r.sort_values('Percent',ascending=False)


## Dataset Preview



In [ ]:
display(df.head()); display(df.tail())

## Dataset Summary



In [ ]:
display(dataset_summary(df))

## Schema



In [ ]:
df.info()

## Summary Statistics



In [ ]:
display(df.describe(include='all').T)

## Missing Value Analysis



In [ ]:
display(missing_report(df))

## Duplicate Analysis



In [ ]:
print(df.duplicated().sum());print(df.duplicated(subset=['company_name','year']).sum())

## Target Distribution



In [ ]:
vc=df['status_label'].value_counts();display(vc);vc.plot(kind='bar',title='Target Distribution');plt.show()

## Year Distribution



In [ ]:
yr=df['year'].value_counts().sort_index();yr.plot(marker='o');plt.title('Records by Year');plt.show()

## Company Analysis



In [ ]:
obs=df.groupby('company_name').size();display(obs.describe());obs.hist();plt.title('Observations per Company');plt.show()

## Numeric Feature Profiling



In [ ]:
num=df.select_dtypes(include='number');display(num.describe().T);

## Histograms



In [ ]:
num.hist(figsize=(18,18));plt.tight_layout();plt.show()

## Boxplots



In [ ]:
for c in num.columns:
    plt.figure(figsize=(6,2.5))
    plt.boxplot(df[c].dropna(),vert=False)
    plt.title(c)
    plt.show()

## Correlation



In [ ]:
corr=num.corr()
fig=plt.figure(figsize=(10,8))
plt.imshow(corr,cmap='coolwarm')
plt.colorbar()
plt.xticks(range(len(corr.columns)),corr.columns,rotation=90)
plt.yticks(range(len(corr.columns)),corr.columns)
plt.title('Correlation Heatmap')
plt.show()

## Alive vs Failed



In [ ]:
for c in num.columns[:8]:
    plt.figure()
    df.boxplot(column=c,by='status_label')
    plt.suptitle('')
    plt.title(c)
    plt.show()

## Outlier Report



In [ ]:
Q1=num.quantile(.25);Q3=num.quantile(.75);IQR=Q3-Q1
out=((num<(Q1-1.5*IQR))|(num>(Q3+1.5*IQR))).sum().sort_values(ascending=False)
display(out.to_frame('Outlier Count'))

## Business Insights



In [ ]:
print('1. Evaluate class imbalance before modeling.')
print('2. Review highly correlated variables before feature selection.')
print('3. Investigate extreme outliers before scaling.')
print('4. Document assumptions and data limitations.')